In [ ]:
import ee
import geemap
from utils.variables import (
    PROJECT,
    TEST_SITE_IDs,
    ANALYSIS_START_YR,
    ANALYSIS_END_YR,
    GLC_CLASSES,
    NFW_THRESHOLD
)

In [ ]:
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
# Data imports
PAs = ee.FeatureCollection("WCMC/WDPA/current/polygons")
OECMs = ee.FeatureCollection("WCMC/WDOECM/current/polygons")
GLC = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")
HGFC = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GPW = ee.ImageCollection("projects/global-pasture-watch/assets/ggc-30m/v1/grassland_c")
NFW = ee.ImageCollection("projects/nature-trace/assets/forest_typology/natural_forest_2020_v1_0_collection")

In [ ]:
# Select test sites
all_sites = (ee.FeatureCollection([PAs, OECMs]).flatten()
             .filter(ee.Filter.eq("REALM", "Terrestrial")))
test_sites = (all_sites.filter(ee.Filter.inList("SITE_ID", TEST_SITE_IDs)))

In [ ]:
# Process Global Land Cover Change data

GLC_mosaic = GLC.filterBounds(test_sites).mosaic()

# Select bands corresponding to analysis years
analysis_years = list[int](range(ANALYSIS_START_YR, ANALYSIS_END_YR + 1))
band_names = [f"b{year - 2000 + 1}" for year in analysis_years]
new_band_names = [f"GLC_{year}" for year in analysis_years]
GLC_filtered = GLC_mosaic.select(band_names, new_band_names)

# Remap class values to 1-36
def remap_classes(band):
    return (GLC_filtered.select(band)
            .remap(GLC_CLASSES, ee.List.sequence(1, len(GLC_CLASSES)), defaultValue=0)
            .rename([band]))

remapped_bands = [remap_classes(band) for band in new_band_names]
GLC_remapped = ee.Image.cat(remapped_bands)

In [ ]:
# Process Global Pasture Watch data
year_strings = [str(year) for year in range(ANALYSIS_START_YR, ANALYSIS_END_YR + 1)]
GPW_filtered = GPW.filter(ee.Filter.inList("system:index", year_strings)).toBands()
GPW_renamed = GPW_filtered.rename([f"GPW_{year}" for year in year_strings])

In [ ]:
# Process Hansen Global Forest Change data
forest_loss = HGFC.select("lossyear").unmask(0)
mask = (forest_loss.gte(ANALYSIS_START_YR - 2000)
        .And(forest_loss.lte(ANALYSIS_END_YR - 2000)))
forest_loss_masked = forest_loss.updateMask(mask)

In [ ]:
# Process Natural Forests of the World data
NFW_mosaic = NFW.filterBounds(test_sites).mosaic()
NFW_thresholded = NFW_mosaic.gte(NFW_THRESHOLD)

In [ ]:
# Create natural cover raster for ANALYSIS_END_YR by creating and inverting a non-natural mask

# Select GLC data for ANALYSIS_END_YR
GLC_current = GLC_remapped.select(f"GLC_{ANALYSIS_END_YR}")

# Select cropland (1-4) and impervious surfaces (30)
anthro_classes = ee.List([1, 2, 3, 4, 30])
anthro_mask = GLC_current.remap(anthro_classes, ee.List.repeat(1, anthro_classes.size()), defaultValue=0)

# Select any forest loss pixels from ANALYSIS_START_YR to ANALYSIS_END_YR using GFW
forest_loss_mask = (forest_loss.gte(ANALYSIS_START_YR - 2000)
                    .And(forest_loss.lte(ANALYSIS_END_YR - 2000)))

# Select any cultivated grassland pixels using GPW
GPW_current = GPW_renamed.select(f"GPW_{ANALYSIS_END_YR}")
pasture_mask = GPW_current.eq(1).unmask(0)

# Select any pixels of "Forest" class that do not overlap with NFW
forest_classes = ee.List([5,6,7,8,9,10,11,12,13,14])
planted_forest_mask = (GLC_current.remap(forest_classes, ee.List.repeat(1, forest_classes.size()), defaultValue=0)
                      .updateMask(NFW_thresholded.eq(0))).unmask(0)

# Combine all non-natural masks and invert; apply mask to landcover raster
non_natural_mask = anthro_mask.add(forest_loss_mask).add(pasture_mask).add(planted_forest_mask)
natural_cover_mask = non_natural_mask.eq(0)
natural_cover = GLC_current.updateMask(natural_cover_mask)

In [ ]:
# Calculate percent natural cover for ANALYSIS_END_YR within a PA

# Select site
test_site = ee.Feature(test_sites.first()).geometry()

# Calculate total area of site
site_area = test_site.area().getInfo()

# Calculate total area of natural cover
natural_cover_area = (ee.Image.pixelArea()
     .updateMask(natural_cover_mask)
     .reduceRegion(ee.Reducer.sum(), test_site, scale=30, maxPixels=1e13)
     .get('area')
     .getInfo())

pct_natural_cover = natural_cover_area / site_area
print(pct_natural_cover)


In [ ]:
# Visualization

from utils.variables import GLC_PALETTE
Map = geemap.Map()

# Map.addLayer(GLC_remapped.select('GLC_2022'), {'min':0, 'max':36, 'palette': GLC_PALETTE}, "GLC")
# Map.addLayer(GPW_renamed.select('GPW_2022').selfMask(), {'min':1, 'max':2, 'palette': ['#ffcd73','#ff9916']}, "GPW")
# Map.addLayer(NFW_thresholded.selfMask(), {'palette': 'green'}, "NFW")
# Map.addLayer(forest_loss_masked, {'min':ANALYSIS_START_YR-2000, 'max':ANALYSIS_END_YR-2000, 'palette': ['yellow', 'red']}, "Forest loss")
# Map.addLayer(test_sites, {"color": "red"}, "Test sites")

# Map.addLayer(anthro_mask, {'min':0, 'max':1, 'palette': ['white', 'red']}, "Anthro Mask")
# Map.addLayer(forest_loss_mask, {'min':0, 'max':1, 'palette': ['white', 'green']}, "Forest loss mask")
# Map.addLayer(pasture_mask, {'min':0, 'max':1, 'palette': ['white', 'blue']}, "Pasture mask")
# Map.addLayer(planted_forest_mask, {'min':0, 'max':1, 'palette': ['white', 'orange']}, "Planted forest mask")
# Map.addLayer(non_natural_mask.selfMask(), {'palette': 'red'}, "Non-natural mask")
# Map.addLayer(natural_cover_mask.selfMask(), {'palette': 'green'}, "Natural cover mask")
Map.addLayer(natural_cover, {'palette': GLC_PALETTE}, "Natural cover")

Map.addLayer(test_site, {"color": "red"}, "Test site")
Map.centerObject(test_site, 7)
Map.setOptions('HYBRID')


Map